In [8]:
import pandas as pd
import numpy as np
from utils.data import load_data
from utils.taxonomy import load_taxonomy
import requests
import json
from openai import OpenAI
from utils.data import load_data, print_random_record
from utils.taxonomy import load_taxonomy

In [9]:
import glob

# Get all .jsonl
jsonl_files = glob.glob("data/labels/*.jsonl")

# Read all .jsonl files and concatenate into a single DataFrame
df_labels = pd.concat([pd.read_json(f, lines=True) for f in jsonl_files], ignore_index=True)
print("Total labels loaded: ", len(df_labels), "with unique ids ", len(df_labels['custom_id'].unique()))

### Expanding the 'response' field in df_labels

# Step 1: Ensure 'response' is a dictionary (if it's a JSON string)
df_labels['response'] = df_labels['response'].apply(
    lambda x: json.loads(x) if isinstance(x, str) else x
)
# Step 2: Flatten the top-level 'response' field
df_response_flat = pd.json_normalize(df_labels['response'])

# Step 3: Extract 'body.choices[0]' safely and normalize it
df_choices_flat = pd.json_normalize(
    df_labels['response'].apply(lambda r: r.get('body', {}).get('choices', [{}])[0]),
    sep='.'
)

# Step 4: Combine everything into one DataFrame
df_combined = pd.concat([df_labels.drop(columns=['response']), df_response_flat, df_choices_flat], axis=1)

## Core Df to refer for labels
labels_core_cols = ["custom_id", "status_code", "body.choices", "message.content"]
df_labels_core = df_combined[labels_core_cols]
df_labels_core.head()

Total labels loaded:  6000 with unique ids  6000


,custom_id,status_code,body.choices,message.content
0,WOS:001468126000008,200,"[{'index': 0, 'message': {'role': 'assistant',...","{\n ""kingdom"": null,\n ""phylum"": null,\n ""c..."
1,WOS:001064605600001,200,"[{'index': 0, 'message': {'role': 'assistant',...","{\n ""kingdom"": [""Animalia""],\n ""phylum"": [""A..."
2,WOS:000424163600009,200,"[{'index': 0, 'message': {'role': 'assistant',...","{\n ""kingdom"": [""Animalia""],\n ""phylum"": [""A..."
3,WOS:000404099800018,200,"[{'index': 0, 'message': {'role': 'assistant',...","{\n ""kingdom"": [""Plantae""],\n ""phylum"": [""Pi..."
4,WOS:000082404900011,200,"[{'index': 0, 'message': {'role': 'assistant',...","{\n ""kingdom"": null,\n ""phylum"": null,\n ""c..."


In [10]:
df, _ = load_data()
core_cols = [
    "Publication Type", "Article Title", "Language", "Document Type",
    "Author Keywords", "Keywords Plus", "Abstract", 
    "Publisher", "Publisher City", "Publication Date", "Publication Year", 
    "Web of Science Index", "WoS Categories", "Research Areas", 
    "Highly Cited Status", "Hot Paper Status", "UT (Unique WOS ID)"
]
df_core = df[core_cols].copy()

Loading dataset from local file...
Dataset shape: (359435, 72)
 Count Summary : {'row_count': 359435, 'web_of_science_record_count': np.int64(359435), 'web_of_science_record_distinct_count': 358933, 'duplicated_row_count': np.int64(502)}
Dropped 502 duplicated rows. Remaining rows: 358933
Dropped 16449 rows without Abstract. Remaining rows: 342484
Sampling 1000 rows from the dataset of length 342484.
 51 Columns with null values:
Cited References            342484
Meeting Abstract            342484
Book Group Authors          342461
Book Series Subtitle        342445
Group Authors               342178
Book Author Full Names      342068
Book Authors                342068
Conference Host             341191
Highly Cited Status         340300
Hot Paper Status            340300
Supplement                  340024
Conference Sponsor          339945
Part Number                 339633
Book DOI                    339045
Book Series Title           338975
Book Editors                337976
ISBN  

In [7]:
df['UT (Unique WOS ID)'].nunique(), df['UT (Unique WOS ID)'].isna().sum()

(342484, np.int64(0))

In [4]:
unique_label_ids = df_labels_core['custom_id'].unique()
core_ids = df_core['UT (Unique WOS ID)']
missing_ids = set(unique_label_ids) - set(core_ids)
if not missing_ids:
    print("Success: All custom_ids from df_labels_core are present in df_core['UT (Unique WOS ID)']")
else:
    print(f"Error: Missing {len(missing_ids)} ids in df_core['UT (Unique WOS ID)']: {list(missing_ids)}")

# Inner join df_core and df_labels_core on UT/custom_id
df_joined = df_core.merge(df_labels_core, left_on='UT (Unique WOS ID)', right_on='custom_id', how='inner', suffixes=('_core', '_labels'))

print("Inner join row count:", len(df_joined))
if len(df_joined) < len(df_labels_core):
    print(f"Warning: {len(df_labels_core) - len(df_joined)} rows lost in inner join.")
else:
    print("No rows lost in inner join.")

Success: All custom_ids from df_labels_core are present in df_core['UT (Unique WOS ID)']
Inner join row count: 6000
No rows lost in inner join.


In [8]:
print_random_record(df_joined)

UT (Unique WOS ID):
WOS:000225974900003
----------------------------------------
Article Title:
Bergmann's rule and the mammal fauna of northern North America
----------------------------------------
Abstract:
The observation that on the whole... larger species live farther north and the
smaller ones farther south was first published by Carl Bergmann in 1847.
However, why animal body mass might show such spatial variation, and indeed
whether it is a general feature of animal assemblages, is currently unclear. We
discuss reasons for this uncertainty, and use Our conclusions to direct an
analysis of Bergmann's rule in the mammals in northern North America, in the
communities of species occupying areas that were covered by ice at the last
glacial maximum. First, we test for the existence of Bergmann's rule in this
assemblage, and investigate whether small- and large-bodied species show
different spatial patterns of body size variation. We then attempt to explain
the spatial variation in t